In [3]:
# ==============================================================================
# 📝 PIPELINE DE EXTRAÇÃO E TRATAMENTO DE DADOS - BIGQUERY COM PYTHON
# Dependências: google-cloud-bigquery, pandas, pyarrow, db-dtypes
# Parâmetros: Projeto 'aula-ebac-1', Dataset Público 'covid19_ecdc_eu'
# ==============================================================================

import os
from google.cloud import bigquery
import pandas as pd

# 1. Inicializa o cliente do BigQuery usando sua autenticação gcloud CLI
client = bigquery.Client(project="aula-ebac-1")
print("✅ Cliente BigQuery autenticado com sucesso!\n")

# 2. Executa a Query 1: Top 20 Países por Casos Confirmados
print("⏳ Executando Query 1 (Top 20 Países)...")
query_top_20 = """
SELECT 
  countries_and_territories AS pais,
  SUM(daily_confirmed_cases) AS total_casos,
  SUM(daily_deaths) AS total_mortes
FROM 
  `bigquery-public-data.covid19_ecdc_eu.covid_19_geographic_distribution_worldwide`
GROUP BY 
  pais
ORDER BY 
  total_casos DESC
LIMIT 20;
"""
df_top_20 = client.query(query_top_20).to_dataframe()
print("▶️ Amostra do DataFrame - Top 20 Países:")
print(df_top_20.head(), "\n" + "="*50 + "\n")

# 3. Executa a Query 2: Evolução Mensal no Brasil (2020)
print("⏳ Executando Query 2 (Evolução Mensal Brasil)...")
query_brasil = """
SELECT 
  EXTRACT(MONTH FROM date) AS mes,
  SUM(daily_confirmed_cases) AS casos_no_mes,
  SUM(daily_deaths) AS mortes_no_mes
FROM 
  `bigquery-public-data.covid19_ecdc_eu.covid_19_geographic_distribution_worldwide`
WHERE 
  countries_and_territories = 'Brazil' 
  AND EXTRACT(YEAR FROM date) = 2020
GROUP BY 
  mes
ORDER BY 
  mes ASC;
"""
df_brasil = client.query(query_brasil).to_dataframe()
print("▶️ Amostra do DataFrame - Brasil 2020:")
print(df_brasil.head(), "\n" + "="*50 + "\n")

# 4. Limpeza e Padronização dos Dados
print("🧹 Aplicando limpeza e padronização...")
df_top_20.columns = df_top_20.columns.str.strip().str.lower()
df_brasil.columns = df_brasil.columns.str.strip().str.lower()

df_top_20['total_casos'] = df_top_20['total_casos'].astype(int)
df_top_20['total_mortes'] = df_top_20['total_mortes'].astype(int)

df_brasil['mes'] = df_brasil['mes'].astype(int)
df_brasil['casos_no_mes'] = df_brasil['casos_no_mes'].astype(int)
df_brasil['mortes_no_mes'] = df_brasil['mortes_no_mes'].astype(int)
print("✅ Limpeza concluída!\n" + "="*50 + "\n")

# 5. Validação de Qualidade de Dados (Exigência do Exercício)
def validar_dataframe(df, nome_base):
    print(f"📋 Validação da Base: {nome_base}")
    print(f"-> Schema / Tipos:\n{df.dtypes}")
    print(f"-> Valores nulos: {df.isnull().sum().sum()}")
    print(f"-> Linhas duplicadas: {df.duplicated().sum()}")
    print(f"-> Total de registros: {len(df)}\n")

validar_dataframe(df_top_20, "Top 20 Países")
validar_dataframe(df_brasil, "Evolução Mensal Brasil")
print("="*50 + "\n")

# 6. Criação das Pastas e Salvamento Estruturado (CSV e Parquet)
print("💾 Salvando arquivos de saída (outputs)...")
os.makedirs("output/csv", exist_ok=True)
os.makedirs("output/parquet", exist_ok=True)

df_top_20.to_csv("output/csv/top_20_paises.csv", index=False)
df_top_20.to_parquet("output/parquet/top_20_paises.parquet", index=False)

df_brasil.to_csv("output/csv/evolucao_mensal_brasil.csv", index=False)
df_brasil.to_parquet("output/parquet/evolucao_mensal_brasil.parquet", index=False)

print("🎉 Processo Concluído com sucesso! Os arquivos estão na pasta 'output'.")


✅ Cliente BigQuery autenticado com sucesso!

⏳ Executando Query 1 (Top 20 Países)...


c:\Users\wellm\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


▶️ Amostra do DataFrame - Top 20 Países:
                       pais  total_casos  total_mortes
0  United_States_of_America     16256754        299177
1                     India      9884100        143355
2                    Brazil      6901952        181402
3                    Russia      2653928         46941
4                    France      2376852         57911 

⏳ Executando Query 2 (Evolução Mensal Brasil)...


c:\Users\wellm\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


▶️ Amostra do DataFrame - Brasil 2020:
   mes  casos_no_mes  mortes_no_mes
0    1             0              0
1    2             1              0
2    3          4578            159
3    4         73583           5307
4    5        420278          23368 

🧹 Aplicando limpeza e padronização...
✅ Limpeza concluída!

📋 Validação da Base: Top 20 Países
-> Schema / Tipos:
pais              str
total_casos     int64
total_mortes    int64
dtype: object
-> Valores nulos: 0
-> Linhas duplicadas: 0
-> Total de registros: 20

📋 Validação da Base: Evolução Mensal Brasil
-> Schema / Tipos:
mes              int64
casos_no_mes     int64
mortes_no_mes    int64
dtype: object
-> Valores nulos: 0
-> Linhas duplicadas: 0
-> Total de registros: 12


💾 Salvando arquivos de saída (outputs)...
🎉 Processo Concluído com sucesso! Os arquivos estão na pasta 'output'.
